<a href="https://colab.research.google.com/github/wilsonmarundaa-netizen/updated-flyRank-intern/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wilsonmarundaa-netizen/updated-flyRank-intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The rule: Pages with declining traffic should be prioritized for review.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
!wget https://raw.githubusercontent.com/wilsonmarundaa-netizen/updated-flyRank-intern/main/data/raw/content_refresh_anonymized.csv

import pandas as pd

df = pd.read_csv('content_refresh_anonymized.csv')



df.info()


--2026-08-19 19:41:30--  https://raw.githubusercontent.com/wilsonmarundaa-netizen/updated-flyRank-intern/main/data/raw/content_refresh_anonymized.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6727670 (6.4M) [text/plain]
Saving to: ‘content_refresh_anonymized.csv’

content_refresh_ano 100%[===================>]   6.42M  --.-KB/s    in 0.02s   

2026-08-19 19:41:30 (336 MB/s) - ‘content_refresh_anonymized.csv’ saved [6727670/6727670]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2  

In [ ]:
df["trend_direction"].unique()

array(['down', 'stable', 'new', 'up', 'flat'], dtype=object)

In [ ]:
df["down_trend"] = df["trend_direction"] == "down"

In [ ]:
df["impressions_90d"].mean()

np.float64(5200.3663)

In [ ]:
import numpy as np

df["impressions_30d_rate"] = df["impressions_last_30d"] / 30
df["impressions_90d_rate"] = df["impressions_90d"] / 90

df["score"] = np.where(
    df["impressions_30d_rate"] > df["impressions_90d_rate"],
    "up",
    "down"
)
df["score"]

,score
0,down
18545,down
18525,down
18526,down
18527,down
18528,down
18529,down
18530,down
18531,down
18533,down


In [ ]:
df = df.sort_values("score", ascending=False).head(20)
df[["content_id","score","impressions_90d","impressions_last_30d","impressions_30d_rate","impressions_90d_rate"]]

,content_id,score,impressions_90d,impressions_last_30d,impressions_30d_rate,impressions_90d_rate
0,content_304f48230142,down,3803,578,19.266667,42.255556
18545,content_ee78be242093,down,196,47,1.566667,2.177778
18543,content_bf7b1ebef464,down,12,3,0.100000,0.133333
18541,content_a87f07530a71,down,39,9,0.300000,0.433333
18540,content_f7c3e75bbcb2,down,14813,2334,77.800000,164.588889
18539,content_f3c49edd9c15,down,20,1,0.033333,0.222222
18538,content_e5fe2f1d01f9,down,1538,126,4.200000,17.088889
18537,content_bc805f67c328,down,26,4,0.133333,0.288889
18536,content_93f8f0b760af,down,1124,183,6.100000,12.488889
18535,content_aed858812e1e,down,607,184,6.133333,6.744444


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Actions that could be taken: refresh, expansion, protection, pruning or monitoring



In [ ]:
import numpy as np

df["action"] = np.select(
    [
        df["score"] == "up",
        df["score"] == "down",

    ],
    [

        "Monitoring/Protection",
        "Pruning/Refresh",
    ],
    default="Monitoring"
)

df[["content_id","score","impressions_90d","action"]]


,content_id,score,impressions_90d,action
0,content_304f48230142,down,3803,Pruning/Refresh
18545,content_ee78be242093,down,196,Pruning/Refresh
18543,content_bf7b1ebef464,down,12,Pruning/Refresh
18541,content_a87f07530a71,down,39,Pruning/Refresh
18540,content_f7c3e75bbcb2,down,14813,Pruning/Refresh
18539,content_f3c49edd9c15,down,20,Pruning/Refresh
18538,content_e5fe2f1d01f9,down,1538,Pruning/Refresh
18537,content_bc805f67c328,down,26,Pruning/Refresh
18536,content_93f8f0b760af,down,1124,Pruning/Refresh
18535,content_aed858812e1e,down,607,Pruning/Refresh


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
df[["content_id","score","impressions_last_30d","impressions_90d","impressions_30d_rate","impressions_90d_rate","trend_direction","action"]]

,content_id,score,impressions_last_30d,impressions_90d,impressions_30d_rate,impressions_90d_rate,trend_direction,action
0,content_304f48230142,down,578,3803,19.266667,42.255556,down,Pruning/Refresh
18545,content_ee78be242093,down,47,196,1.566667,2.177778,down,Pruning/Refresh
18543,content_bf7b1ebef464,down,3,12,0.100000,0.133333,down,Pruning/Refresh
18541,content_a87f07530a71,down,9,39,0.300000,0.433333,down,Pruning/Refresh
18540,content_f7c3e75bbcb2,down,2334,14813,77.800000,164.588889,down,Pruning/Refresh
18539,content_f3c49edd9c15,down,1,20,0.033333,0.222222,down,Pruning/Refresh
18538,content_e5fe2f1d01f9,down,126,1538,4.200000,17.088889,down,Pruning/Refresh
18537,content_bc805f67c328,down,4,26,0.133333,0.288889,stable,Pruning/Refresh
18536,content_93f8f0b760af,down,183,1124,6.100000,12.488889,down,Pruning/Refresh
18535,content_aed858812e1e,down,184,607,6.133333,6.744444,down,Pruning/Refresh


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.